# Particle Filtering for Apollo-Style Tracking

## Historical problem

The Kalman filter of 1960 made recursive state estimation practical: as new measurements arrived, beliefs about an unseen moving state could be updated sequentially. Particle filters extended that logic to nonlinear or non-Gaussian models.

This notebook uses an **Apollo-style** tracking problem rather than a historically exact Apollo navigation model. A moving object follows hidden dynamics in two dimensions, and an observer at the origin receives noisy range and bearing measurements. The goal is to infer the hidden trajectory over time.

## Model sketch

State:

$$
x_t = (p_{x,t}, p_{y,t}, v_{x,t}, v_{y,t})
$$

Dynamics:

$$
x_t = A x_{t-1} + \varepsilon_t
$$

Observation:

$$
y_t = (\text{range}_t, \text{bearing}_t) + \eta_t
$$

where the observation map is nonlinear because range and bearing depend on the hidden position through square roots and arctangents.

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve().parents[0]
SHARED = ROOT / "00_shared"
if str(SHARED) not in sys.path:
    sys.path.append(str(SHARED))

from plotting import save_fig, set_plot_style

set_plot_style()
rng = np.random.default_rng(123)

## Simulate a hidden trajectory and noisy observations

In [ ]:
dt = 1.0
T = 60
range_sd = 6.0
bearing_sd = 0.025
process_sd = 0.55

A = np.array(
    [
        [1.0, 0.0, dt, 0.0],
        [0.0, 1.0, 0.0, dt],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ]
)

Q = process_sd**2 * np.array(
    [
        [0.25, 0.0, 0.5, 0.0],
        [0.0, 0.25, 0.0, 0.5],
        [0.5, 0.0, 1.0, 0.0],
        [0.0, 0.5, 0.0, 1.0],
    ]
)


def wrap_angle(angle):
    return (angle + np.pi) % (2.0 * np.pi) - np.pi


def observe(state):
    x, y = state[..., 0], state[..., 1]
    rng_val = np.sqrt(x**2 + y**2)
    bearing = np.arctan2(y, x)
    return np.stack([rng_val, bearing], axis=-1)


truth = np.zeros((T, 4))
truth[0] = np.array([30.0, 20.0, 2.2, 1.3])
for t in range(1, T):
    truth[t] = rng.multivariate_normal(A @ truth[t - 1], Q)

obs = observe(truth)
obs[:, 0] += rng.normal(0.0, range_sd, size=T)
obs[:, 1] = wrap_angle(obs[:, 1] + rng.normal(0.0, bearing_sd, size=T))

print("Truth and observations simulated.")

## Bootstrap particle filter

Each step repeats the usual sequential Bayes logic:

1. propagate particles through the dynamics,
2. weight them by the observation likelihood,
3. normalise,
4. resample when necessary.

The particle cloud is therefore a moving approximation to the filtering distribution.

In [ ]:
def systematic_resample(weights, rng):
    n = len(weights)
    positions = (rng.random() + np.arange(n)) / n
    cumulative = np.cumsum(weights)
    indexes = np.zeros(n, dtype=int)
    i = 0
    j = 0
    while i < n:
        if positions[i] < cumulative[j]:
            indexes[i] = j
            i += 1
        else:
            j += 1
    return indexes


def log_likelihood(y, particles):
    pred = observe(particles)
    range_resid = (y[0] - pred[:, 0]) / range_sd
    bearing_resid = wrap_angle(y[1] - pred[:, 1]) / bearing_sd
    return -0.5 * (range_resid**2 + bearing_resid**2)


def run_particle_filter(obs, n_particles=1500):
    particles = np.zeros((n_particles, 4))
    particles[:, 0] = rng.normal(20.0, 15.0, size=n_particles)
    particles[:, 1] = rng.normal(15.0, 15.0, size=n_particles)
    particles[:, 2] = rng.normal(2.0, 1.5, size=n_particles)
    particles[:, 3] = rng.normal(1.0, 1.5, size=n_particles)
    weights = np.full(n_particles, 1.0 / n_particles)

    means = np.zeros((T, 4))
    ess = np.zeros(T)
    snapshots = {}
    snapshot_times = {5, 20, 40, 59}

    for t in range(T):
        if t > 0:
            propagated = np.array([rng.multivariate_normal(A @ p, Q) for p in particles])
            particles = propagated

        logw = log_likelihood(obs[t], particles)
        logw = logw - np.max(logw)
        w = np.exp(logw)
        weights = w / np.sum(w)
        means[t] = np.average(particles, axis=0, weights=weights)
        ess[t] = 1.0 / np.sum(weights**2)

        if t in snapshot_times:
            snapshots[t] = (particles.copy(), weights.copy())

        if ess[t] < n_particles / 2:
            idx = systematic_resample(weights, rng)
            particles = particles[idx]
            weights = np.full(n_particles, 1.0 / n_particles)

    return means, ess, snapshots


pf_means, ess, snapshots = run_particle_filter(obs, n_particles=1500)
print("Particle filter run complete.")

## Sequential Bayes in pictures

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(truth[:, 0], truth[:, 1], color="#111111", lw=2, label="True trajectory")
ax.plot(pf_means[:, 0], pf_means[:, 1], color="#d62728", lw=2, label="PF mean")
ax.scatter(0, 0, color="#1f77b4", s=80, marker="*", label="Observer")
ax.set_title("Hidden trajectory versus particle-filter estimate")
ax.set_xlabel("x position")
ax.set_ylabel("y position")
ax.legend()
fig.tight_layout()
save_fig(fig, Path("figs") / "pf_trajectory.png")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, t in zip(axes.flat, sorted(snapshots)):
    pts, w = snapshots[t]
    ax.scatter(pts[:, 0], pts[:, 1], s=6, alpha=0.15, color="#4c78a8")
    ax.scatter(truth[t, 0], truth[t, 1], color="#d62728", s=50, label="Truth")
    ax.scatter(pf_means[t, 0], pf_means[t, 1], color="#f58518", s=40, label="PF mean")
    ax.scatter(0, 0, color="#1f77b4", s=60, marker="*")
    ax.set_title(f"Particle cloud at t = {t}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
axes[0, 0].legend(loc="upper left")
fig.tight_layout()
save_fig(fig, Path("figs") / "particle_cloud_snapshots.png")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ess, color="#54a24b", lw=2)
ax.axhline(1500 / 2, color="#d62728", ls="--", label="Resampling threshold")
ax.set_title("Effective sample size over time")
ax.set_xlabel("Time")
ax.set_ylabel("ESS")
ax.legend()
fig.tight_layout()
save_fig(fig, Path("figs") / "particle_filter_ess.png")
plt.show()

## Interpretation

This is the essence of sequential Bayesian computation:

- the posterior is updated one observation at a time,
- uncertainty is represented by a particle cloud rather than one single estimate,
- nonlinear observation models remain manageable.

The state estimate stabilises as evidence accumulates, while the particle cloud reveals how uncertainty changes in time.

## References

- Kalman (1960), *A New Approach to Linear Filtering and Prediction Problems*.
- Gordon, Salmond, and Smith (1993), *Novel Approach to Nonlinear/Non-Gaussian Bayesian State Estimation*.
- Doucet, de Freitas, and Gordon (2001), *Sequential Monte Carlo Methods in Practice*.